In [0]:
ALTER TABLE silver.silver_dim_customers
ALTER COLUMN emirates_id
SET TAGS ('pii_category' = 'national_id');
--
ALTER TABLE silver.silver_dim_customers
ALTER COLUMN email
SET TAGS ('pii_category' = 'email');
--
ALTER TABLE silver.silver_dim_customers
ALTER COLUMN phone
SET TAGS ('pii_category' = 'phone');
--
ALTER TABLE gold.gold_dim_customers
ALTER COLUMN emirates_id
SET TAGS ('pii_category' = 'national_id');
--
ALTER TABLE gold.gold_dim_customers
ALTER COLUMN email
SET TAGS ('pii_category' = 'email');
--
ALTER TABLE gold.gold_dim_customers
ALTER COLUMN phone
SET TAGS ('pii_category' = 'phone');
--

CREATE OR REPLACE FUNCTION alwaha_banking_dev_001.governance.governed_mask(col_val STRING, pii_category STRING)
RETURNS STRING
RETURN CASE
    
    When pii_category = 'national_id' THEN CONCAT('***-', RIGHT(col_val, 4))
    When pii_category = 'email' THEN CONCAT(substr(col_val, 1, 3), '***@***', substr(col_val, instr(col_val, '@') + 1))
    When pii_category  = 'phone' THEN CONCAT(substr(col_val, 1, 4), '******', RIGHT(col_val, 2))
    ELSE '***MASKED***'
END;

CREATE POLICY pii_masking_policy
ON CATALOG alwaha_banking_dev_001
COLUMN MASK alwaha_banking_dev_001.governance.governed_mask
TO `account users`
EXCEPT `data_engineers`, `compliance_officer`
FOR TABLES
MATCH COLUMNS
    has_tag('pii_category') AS tagged_col
ON COLUMN tagged_col
USING COLUMNS (
    get_column_tag_value(tagged_col,'pii_category')
);
